**GroupDNA:**  WhatsApp Chat Analysis

**Name:** Lakshita S

Data Science Program June 2026 Batch

26.06.26



## **Feature 1 - The Chat Parser**

### **What it does:**
This is the foundation of the entire project. It opens the WhatsApp export file, reads it line by line, and pulls out three things from each valid message — the timestamp, the sender's name, and the message text. Lines that are system notifications, deleted messages, or media stubs get counted separately and skipped. Every real message gets stored as a dictionary in a list called `msgs`, which every other feature depends on.

### **Concepts used:**
* `open()` and `.read()` to load the file into memory.
* `.split()` to break each line at `" - "` and `": "` to separate timestamp, sender, and message.
* A `for` loop with `if` conditions to detect and skip system messages, media, and deleted entries.

In [ ]:
file_path = '/content/hostel_bois.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    chat = f.read()


lines=chat.split("\n")

def parse_chat(lines):
    msgs=[]
    sys=0
    media=0
    deleted=0

    for line in lines:

        if line=="":
            continue

        if "- " not in line:
            sys=sys+1
            continue

        part1=line.split(" - ",1)
        tstamp=part1[0]
        data=part1[1]

        if ": " not in data:
            sys=sys+1
            continue

        part2=data.split(": ",1)
        sender=part2[0]
        msg=part2[1]

        if msg=="<Media omitted>":
            media=media+1

        if msg=="This message was deleted":
            deleted=deleted+1

        d={}
        d["tstamp"]=tstamp
        d["sender"]=sender
        d["msg"]=msg

        msgs.append(d)

    return msgs,sys,media,deleted

msgs,sys,media,deleted=parse_chat(lines)

users=[]

for i in msgs:
    if i["sender"] not in users:
        users.append(i["sender"])

days=[]

for i in msgs:
    tstamp=i["tstamp"]
    part3=tstamp.split(",")
    date=part3[0]

    if date not in days:
        days.append(date)

print("="*50)
print("FEATURE 1 - CHAT PARSER")
print("="*50)

print("Successfully parsed",len(msgs),"messages from",
      len(users),"participants \n  over",len(days),"days, skipped",sys,
      "system messages,",media,"media-omitted,",deleted,"deleted messages.")

#i used chatgpt for understanding the use of dictionaries and to find errors in my own code

FEATURE 1 - CHAT PARSER
Successfully parsed 3174 messages from 6 participants 
  over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.


## **Feature 2 - Group Overview**

### **What it does:**
This gives a quick summary of the entire chat — who's in the group, how long the chat has been active, and how many messages each person has sent. The participants are ranked from most to least active so you can instantly see who runs the conversation and who barely shows up.

### **Concepts used:**
* A dictionary to count messages per person as we loop through `msgs`.
* `sorted()` with a lambda to rank people by message count.
* `msgs[0]` and `msgs[-1]` to get the first and last message dates without any extra calculation.


In [ ]:
total_msgs=len(msgs)
first=msgs[0]["tstamp"]
last=msgs[-1]["tstamp"]
part1=first.split(",")[0]
part2=last.split(",")[0]
total_days=len(days)
users=[]
count={}
for i in msgs:
    sender=i["sender"]
    if sender not in users:users.append(sender)
    if sender in count:count[sender]=count[sender]+1
    else:count[sender]=1
items=list(count.items())
sorted_users=sorted(items,key=lambda x:x[1],reverse=True)
print("="*50)
print("GROUP OVERVIEW")
print("="*50)
print("Total Messages :",total_msgs)
print("Participants :",len(users))
print("Period :",part1,"to",part2)
print("Total Days :",total_days)
print()
print("Message Count Per Person")
print("-------------------------")
for i in sorted_users:print(i[0],":",i[1])

GROUP OVERVIEW
Total Messages : 3174
Participants : 6
Period : 01/04/24 to 30/05/24
Total Days : 60

Message Count Per Person
-------------------------
Rahul : 953
Priya : 718
Neha : 635
Aman : 490
Karan : 354
Vikas : 24


## **Feature 3 - Peak Activity Timings**

### **What it does:**
Finds the single busiest day in the chat's history and the hour of day when the group is most active on average. This tells you when the group comes alive — whether it's late evenings, lunch breaks, or random midnight bursts.

### **Concepts used:**
* Two dictionaries — one counting messages per date, one per hour.
* `max()` with a key to find the date and hour with the highest count.
* Basic string splitting to extract the date and hour from each timestamp.

In [ ]:
# 1. Create empty dictionaries to keep track of counts
day_counts={}
hour_counts={}

for i in msgs:
    tstamp=i["tstamp"]

    part3=tstamp.split(",")
    date=part3[0].strip()
    if date in day_counts:
      day_counts[date]=day_counts[date]+1
    else:
      day_counts[date]=1

    time_part=part3[1].strip()
    hour=time_part.split(":")[0]
    if hour in hour_counts:
      hour_counts[hour]=hour_counts[hour]+1
    else:
      hour_counts[hour]=1

#Find the busiest day using max()
busiest_day=max(day_counts,key=lambda x:day_counts[x])
max_day_messages=day_counts[busiest_day]

busiest_hour_start= max(hour_counts,key=lambda x:hour_counts[x])
max_hour_messages  =hour_counts[busiest_hour_start]

# Format the hour block
next_hour=int(busiest_hour_start)+1
busiest_hour_string=f"{busiest_hour_start}:00 - {next_hour}:00"
avg_messages_per_day=round(max_hour_messages/total_days)

print("Busiest day  :",busiest_day,f"({max_day_messages} messages)")
print("Busiest hour :",busiest_hour_string,f"(avg {avg_messages_per_day} messages per day)")

Busiest day  : 04/05/24 (76 messages)
Busiest hour : 18:00 - 19:00 (avg 4 messages per day)


## **Feature 4 - Spatial Activity Heatmap Matrix**

### **What it does:**
This module builds a text-based, scaled visual grid mapping user engagement against a 24-hour day. By grouping hours into 3-hour chunks, it prints character symbols (`.`, `░`, `▒`, `█`) that clearly display whether a person chats mostly in the early morning, afternoon, or deep in the middle of the night.

### **Concepts used:**
* **NumPy multi-dimensional arrays (`np.zeros`)** to set up a cross-referenced coordinate matrix (Rows = Users, Columns = 24 Hours).
* Grid step iteration (`range(0, 24, 3)`) to cleanly bucket data into uniform morning, afternoon, and night time slots.
* Relative scaling calculations to convert message numbers into character weights based on an individual's personal maximum activity.

In [ ]:
import numpy as np
users = []

for i in msgs:
    if i["sender"] not in users:
        users.append(i["sender"])
rows =len(users)
cols= 24 #24 hrs

#creating matrix
mat = np.zeros((rows, cols), dtype=int)

#fill matrix
for i in msgs:
    sender= i["sender"]
    tstamp= i["tstamp"]
    part1=tstamp.split(",")[1]
    time=part1.strip()
    hour= int(time.split(":")[0])
    r =users.index(sender)
    mat[r][hour] = mat[r][hour] + 1


# printing heatmap
print("ACTIVITY HEATMAP (messages by hour)")
print()
hours = ["00","03","06","09","12","15","18","21"]
print("      ", end="")
for h in hours:
    print(h, end=" ")
print()
for i in range(rows):

    print(f"{users[i][:6]:<7}", end="")

    # collect all 8 block totals first(used ai as my logic had many errors)
    block_totals=[]
    for j in range(0, 24, 3):
        val=mat[i][j]+mat[i][j+1]+mat[i][j+2]
        block_totals.append(val)

#now im calculating max from the blocks,not individual hours
    maxv= max(block_totals)
    for val in block_totals:
        if maxv == 0:
            ratio = 0
        else:
            ratio = val / maxv
        if ratio <= 0.25:
            ch = ". "
        elif ratio <= 0.50:
            ch = " ░"
        elif ratio <= 0.75:
            ch = " ▒"
        else:
            ch = " █"
        print(ch, end=" ")
    print()

ACTIVITY HEATMAP (messages by hour)

      00 03 06 09 12 15 18 21 
Rahul  .  .   ░  ░  ▒  █  █  █ 
Priya  .  .   ░  █  █  ▒  ▒  ░ 
Karan  .  .  .   ▒  █  █  █  ░ 
Neha   .  .   ░  █  ▒  ▒  █  ▒ 
Aman    █  █ .  .  .  .  .   ░ 
Vikas  .  .   █  ░  █  █  █  █ 


## **Feature 5 - Vocabulary Tracking (Top Group Words)**

### **What it does:**
This component extracts the thematic language of the chat room. It filters out routine conversational filler items (stop words like "the", "a", "is", "lol") and removes formatting noise like commas or punctuation to identify the top 10 substantive words the group relies on most.

### **Concepts used:**
* Text normalization (`.lower()`) to prevent capitalization from splitting identical word counts.
* Punctuation stripping (`.strip()`) to cleanly separate raw words from trailing symbols.
* List slicing (`[:10]`) to filter out lower frequency vocabulary noise and isolate the top results.

In [ ]:
punctuation = '.,!?()[]{}:;\'""-/\\@#*~`'
stop_words =['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
              'it', 'my', 'you', 'me', 'he', 'she', 'we', 'they', 'that', 'this',
              'was', 'are', 'be', 'as', 'at', 'an', 'but', 'so', 'if', 'do', 'not',
              'with', 'have', 'had', 'has', 'can', 'will', 'just', 'no', 'by', 'up',
              'what', 'your', 'from', 'am', 'were', 'its', 'too', 'very', 'well',
              'im', 'dont', 'ive', 'ok', 'okay', 'yeah', 'yes', 'oh', 'lol']

word_counts ={}

for i in msgs:
    msg = i["msg"]
    if msg=="<Media omitted>" or msg== "This message was deleted":
        continue
    words=msg.lower().split()

    for word in words:
        word = word.strip(punctuation)

        if word == "":
            continue
        if word in stop_words:
            continue

        if word in word_counts:
            word_counts[word]= word_counts[word] + 1
        else:
            word_counts[word]=1

top10=sorted(word_counts.items() ,key=lambda x: x[1], reverse=True)[:10]
max_count = top10[0][1]

print("THIS GROUP'S FAVOURITE WORDS")
print()

for word, count in top10:
    bar_len = round((count / max_count) * 20)
    bar = "█" * bar_len
    print(word, bar, count)

THIS GROUP'S FAVOURITE WORDS

how ████████████████████ 321
guys ████████████████████ 318
about █████████████████ 274
hai █████████████████ 268
today ████████████████ 257
his ██████████████ 217
which █████████████ 202
everyone ████████████ 187
telling ███████████ 179
bhai ██████████ 160


## **Feature 6 - Interaction Dynamics (Response Times & Silent Streaks)**

### **What it does:**
This feature dives into the social rhythms of the group by examining two complex metrics:
1. **Average Response Times:** Measures how many minutes on average a user takes to reply when a different person sends a message before them.
2. **Longest Silent Streaks:** Calculates the maximum number of consecutive calendar days an individual goes without sending a single text message to the group.

### **Concepts used:**
* Python's `datetime` module (`datetime.strptime`) to transform raw timestamp strings into object instances capable of subtracting time.
* Time deltas (`.total_seconds()` and `timedelta(days=1)`) to run structural chronological loops and evaluate missing activity days.

In [ ]:
from datetime import datetime, timedelta

#Average Response Time
gaps={}

for p in users:gaps[p]=[]
for idx in range(1,len(msgs)):

    prev=msgs[idx-1]
    curr=msgs[idx]
    if curr["sender"]!=prev["sender"]:
        t1=datetime.strptime(prev["tstamp"],"%d/%m/%y, %H:%M")
        t2=datetime.strptime(curr["tstamp"],"%d/%m/%y, %H:%M")

        diff=(t2-t1).total_seconds()
        if diff>0 and diff<86400: # ignoring gaps longer than 24 hrs
            gaps[curr["sender"]].append(diff)
avg_times={}
for p in users:
    if len(gaps[p])>0:avg_times[p]=sum(gaps[p])/ len(gaps[p])
sorted_avg=sorted(avg_times.items(), key=lambda x:x[1])

print("RESPONSE PATTERNS")
print()

fastest=sorted_avg[0]
slowest=sorted_avg[-1]

def format_time(seconds):
    if seconds<3600:return str(round(seconds/60,1))+" minutes"
    else:return str(round(seconds/3600,1))+" hours"

print("Fastest replier :", fastest[0],f"({format_time(fastest[1])})")
print("Slowest replier :",slowest[0], f"({format_time(slowest[1])})")


#Silent Streaks
first_date=datetime.strptime(msgs[0]["tstamp"],"%d/%m/%y, %H:%M").date()
last_date=datetime.strptime(msgs[-1]["tstamp"],"%d/%m/%y, %H:%M").date()
total_days = (last_date - first_date).days + 1

active_days={}
for p in users:active_days[p]=[]
for m in msgs:
    d=datetime.strptime(m["tstamp"],"%d/%m/%y, %H:%M").date()
    if d not in active_days[m["sender"]]:active_days[m["sender"]].append(d)

streaks={}

for p in users:
    max_streak=0
    best_start =None
    best_end=None
    current =0
    current_start=None
    d=first_date
    while d<= last_date:
        if d not in active_days[p]:
            if current== 0:current_start=d
            current=current+1
            if current >max_streak:
                max_streak =current
                best_start =current_start
                best_end =d
        else:
            current=0
            current_start=None
        d=d+timedelta(days=1)
    streaks[p]=  (max_streak,best_start,best_end)
sorted_streaks= sorted(streaks.items(),key=lambda x:x[1][0],reverse=True)
print()
print("LONGEST SILENT STREAKS (consecutive days with zero messages)")

for p,(streak,start,end) in sorted_streaks:
    if streak ==0:print(p,": 0 days (never went silent)")
    elif streak ==1:print(p,": 1 day")
    else:print(p,":",streak,"days (",start.strftime("%d %b"),"to",end.strftime("%d %b"),")")
#used chatgpt for datetime mod and to debugg errors

RESPONSE PATTERNS

Fastest replier : Rahul (36.5 minutes)
Slowest replier : Aman (56.9 minutes)

LONGEST SILENT STREAKS (consecutive days with zero messages)
Vikas : 11 days ( 23 Apr to 03 May )
Rahul : 0 days (never went silent)
Priya : 0 days (never went silent)
Karan : 0 days (never went silent)
Neha : 0 days (never went silent)
Aman : 0 days (never went silent)


## **Feature 7 - Behavioral Profiling (Personality Archetypes)**

### **What it does:**
This module evaluates each group member's messaging behavior and assigns them exactly one personality archetype based on quantitative scoring rules. Every person is scored across all 9 archetypes and receives the one they score highest on — so no two people share the same label unless their scores are identical.

### **Archetypes covered:**
9 archetypes in total — The Ghost (rarely shows up), The Spammer (sends messages in bursts),
The Night Owl (active after 11 PM), The Storyteller (writes long messages), The Drama Queen
(all-caps or over-exclaims), The Group Mom (caring and responsible), The Comedian (always
laughing), The Question Master (asks the most questions), and The Motivator ✦ (hypes everyone
up) — our bonus archetype.

### **Concepts used:**
* Plain dictionaries to store a score for every archetype per person.
* One scoring block per archetype — each produces a numeric value, not a yes/no.
* `max()` with a custom key to find the highest-scoring archetype per person.
* Loops, conditionals, and string methods (`.lower()`, `.endswith()`, `.isupper()`, `.count()`) — no imports beyond what's already loaded.


### **Bonus archetype — THE MOTIVATOR:**
Detects the group hype person based on energy slang common in Indian college chats (`fire`, `let's go`, `legend`, `king`, `bro`, `beast`, `goat`, `sigma`). The person with the highest percentage of messages containing these words gets this label if no stronger archetype claim exists.

In [ ]:
hype_words = ['fire', "let's go", 'legend', 'king', 'bro', 'lets go', 'beast', 'goat', 'sigma']
caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
                   'please', 'reminder', 'drink water', "don't forget"]
laugh_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
#for findi
print("PERSONALITY ARCHETYPES")
print()
assigned = {}
all_scores = {}  #{archetype_name:score }

for p in users:
    # collect this person's messages
    p_msgs = []
    for m in msgs:
        if m["sender"] == p:
            p_msgs.append(m)

    total = len(p_msgs)
    if total == 0:
        continue

    scores = {}

#SPAMMER: avg consecutive burst
    # count how many times p sends back-to-back before someone else replies
    runs =[]
    run=0
    for m in msgs:
        if m["sender"]==p:
            run=run + 1
        else:
            if run> 0:
                runs.append(run)
                run = 0
    if run > 0:
        runs.append(run)

    avg_run = sum(runs) / len(runs) if runs else 0
    scores["THE SPAMMER"] = avg_run

#GROUP MOM: caring keyword count
    caring_hits = 0
    for m in p_msgs:
        text = m["msg"].lower()
        for word in caring_keywords:
            if word in text:
                caring_hits = caring_hits + 1
                break  # count each message only once
    scores["THE GROUP MOM"] = caring_hits / total * 100

#NIGHT OWL: % messages between 11pm to 5am
    night_count = 0
    for m in p_msgs:
        hour = int(m["tstamp"].split(",")[1].strip().split(":")[0])
        if hour >= 23 or hour <= 4:
            night_count = night_count + 1
    scores["THE NIGHT OWL"] = night_count / total * 100

#STORYTELLER: avg words per message
    total_words = 0
    for m in p_msgs:
        # skipping media and deleted as they skew the countts
        if m["msg"] == "<Media omitted>" or m["msg"] == "This message was deleted":
            continue
        total_words = total_words + len(m["msg"].split())
    scores["THE STORYTELLER"] = total_words / total

#DRAMA QUEEN: % messages that are all-caps OR have 2+ exclamation marks
    #brief rule: exclude short messages under 3 characters
    drama_count = 0
    for m in p_msgs:
        text=m["msg"]
        if len(text) <= 3:
            continue
        is_all_caps = text.isupper()
        has_exclaim = text.count("!") >= 2
        if is_all_caps or has_exclaim:
            drama_count= drama_count + 1
    scores["THE DRAMA QUEEN"] = drama_count / total * 100

#GHOST: silent on more than 60% of days
    days_active = len(active_days[p])
    days_silent = total_days - days_active
    scores["THE GHOST"]= days_silent / total_days * 100

#COMEDIAN: % of mesages containing lol,lmao,haha,etc
    laugh_count = 0
    for m in p_msgs:
        text = m["msg"].lower()
        for word in laugh_words:
            if word in text:
                laugh_count = laugh_count+ 1
                break
    scores["THE COMEDIAN"] = laugh_count / total * 100
#QUESTION MASTER: % of messages ending with ?
    q_count=0
    for m in p_msgs:
        if m["msg"].strip().endswith("?"):
            q_count= q_count + 1
    scores["THE QUESTION MASTER"] = q_count / total * 100

#MOTIVATOR (bonus): % of messages with hypewords
    hype_count = 0
    for m in p_msgs:
        text =m["msg"].lower()
        for word in hype_words:
            if word in text:
                hype_count = hype_count+1
                break
    scores["THE MOTIVATOR"] = hype_count/ total * 100
    all_scores[p] = scores

thresholds = {
    "THE SPAMMER":        3,    # avg burst > 3
    "THE GROUP MOM":      0,    # whoever has most caring keywords
    "THE NIGHT OWL":      60,   # > 60% msgs at night
    "THE STORYTELLER":    30,   # avg words per msg > 30
    "THE DRAMA QUEEN":    30,   # > 30% all-caps messages
    "THE GHOST":          60,   # silent > 60% of days
    "THE COMEDIAN":       0,    # whoever laughs most
    "THE QUESTION MASTER":25,   # > 25% msgs end with ?
    "THE MOTIVATOR":      0,    # whoever hypes most
}

# for each archetype, find the person with the highest score
archetype_winner = {}
for arch in thresholds:
    best_person = None
    best_score = -1
    for p in users:
        score = all_scores[p][arch]
        if score > thresholds[arch] and score > best_score:
            best_score = score
            best_person = p
    if best_person:
        archetype_winner[arch] = best_person
final_archetype = {}  # person -> archetype

#go through archetypes in priority order(main 6 first,tiebreakers last)
priority_order = [
    "THE GHOST",
    "THE SPAMMER",
    "THE NIGHT OWL",
    "THE STORYTELLER",
    "THE DRAMA QUEEN",
    "THE GROUP MOM",
    "THE MOTIVATOR",
    "THE COMEDIAN",
    "THE QUESTION MASTER",
]

for arch in priority_order:
    # find person with highest score for this archetype who isn't assigned yet
    best_person = None
    best_score = -1
    for p in users:
        if p in final_archetype:
            continue
        score = all_scores[p][arch]
        if score>thresholds[arch] and score> best_score:
            best_score = score
            best_person = p
    if best_person:
        final_archetype[best_person]= (arch, best_score)

#anyone still unassigned gets the archetype they scored highest on
for p in users:
    if p not in final_archetype:
        best_arch = max(all_scores[p], key=lambda a: all_scores[p][a])
        final_archetype[p] = (best_arch, all_scores[p][best_arch])

for p in users:
    arch, score = final_archetype[p]

    if arch == "THE SPAMMER":
        detail = "avg " + str(round(score, 1)) + " msgs in a row"
    elif arch == "THE GHOST":
        detail = str(round(score, 1)) + "% of days silent"
    elif arch == "THE STORYTELLER":
        detail = "avg " + str(round(score, 1)) + " words per msg"
    elif arch == "THE NIGHT OWL":
        detail = str(round(score, 1)) + "% msgs after 11 PM"
    elif arch == "THE DRAMA QUEEN":
        detail = str(round(score, 1)) + "% all-caps messages"
    elif arch == "THE GROUP MOM":
        detail = str(round(score, 1)) + "% caring messages"
    elif arch == "THE COMEDIAN":
        detail = str(round(score, 1)) + "% msgs have laughing"
    elif arch == "THE QUESTION MASTER":
        detail = str(round(score, 1)) + "% msgs end with ?"
    elif arch == "THE MOTIVATOR":
        detail = str(round(score, 1)) + "% msgs have hype words"
    else:
        detail = ""

    print(p, "→", arch, "(" + detail + ")")
    #used claude for certain code and to understand them , also to find errors

PERSONALITY ARCHETYPES

Rahul → THE SPAMMER (avg 4.5 msgs in a row)
Priya → THE GROUP MOM (63.9% caring messages)
Karan → THE STORYTELLER (avg 55.6 words per msg)
Neha → THE DRAMA QUEEN (62.2% all-caps messages)
Aman → THE NIGHT OWL (79.8% msgs after 11 PM)
Vikas → THE GHOST (73.3% of days silent)


## **Feature 8 - The Final Integrated Receipt Report**

### **What it does:**
The final component aggregates every value calculated across the entire project and structures the output inside a unified report window. It acts as a final execution check, adjusting the display layout so the final summary dashboard resembles a perfectly aligned print receipt.

### **Concepts used:**
* Advanced **f-string alignment modifiers** (`{var:<26}`, `{var:>25}`) to define consistent cell padding widths across all variable lengths.
* Memory state sharing to extract variables created in previous cells without repeating calculation scripts.

In [ ]:
# build display strings from Feature 6 variables
def format_time(seconds):
    if seconds< 3600:
        return str(round(seconds/60,1)) + " mins"
    return str(round(seconds/3600, 1)) + " hrs"

fastest_str = sorted_avg[0][0] + " (" + format_time(sorted_avg[0][1]) + ")"
slowest_str = sorted_avg[-1][0] + " (" + format_time(sorted_avg[-1][1]) + ")"

streak_results = []
for p in users:
    streak_val = streaks[p][0]
    if streak_val == 0:
        streak_str = "Never silent"
    else:
        streak_str = str(streak_val) + " days"
    streak_results.append((p, streak_str, streak_val))

streak_results = sorted(streak_results, key=lambda x: x[2], reverse=True)

# first try to find it from system messages in the chat
# if nothing found in messages, use the filename
group_name = None
for m in msgs:
    text = m["msg"]
    t = text.lower()
    if "created group" in t or "created this group" in t:
        if '"' in text:
            group_name = text.split('"')[1]
            break
        elif "'" in text:
            group_name = text.split("'")[1]
            break
    elif "subject was changed to" in t:
        group_name = text.split("subject was changed to")[1].strip().strip('"')
        break

#if nothing found, extract name from the file_path variable set in Feature 1
if group_name is None:
    file_name = file_path.split("/")[-1]
    file_name = file_name.replace(".txt", "")
    file_name = file_name.replace("_", " ")
    group_name = file_name.title()

WIDTH = 57

print("┌" + "─" * (WIDTH - 2) + "┐")
print(f"│{'GROUPDNA':^{WIDTH-2}}│")
print(f"│{'Your WhatsApp group chat, decoded.':^{WIDTH-2}}│")
print(f"│{'Spotify Wrapped, but for your friend group!':^{WIDTH-2}}│")

print(f"│{'WHATSAPP CHAT ANALYTICS RECEIPT':^{WIDTH-2}}│")
print(f"│{'-' * 35:^{WIDTH-2}}│")
print(f"│     GROUP NAME : {group_name:<36} │")
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│  {'METRIC / ITEM':<26}{'VALUE / STATS':>25}  │")
print("├" + "─" * (WIDTH - 2) + "┤")

# Core Stats
print(f"│  {'Total Messages':<26}{len(msgs):>25}  │")
print(f"│  {'Total Participants':<26}{len(users):>25}  │")
print(f"│  {'Total Active Days':<26}{len(days):>25}  │")
print(f"│  {'Busiest Day':<26}{busiest_day:>25}  │")
print(f"│  {'Busiest Hour Block':<26}{busiest_hour_string:>25}  │")

# Response Patterns
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│{'RESPONSE PATTERNS':^{WIDTH-2}}│")
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│  {'Fastest Replier':<26}{fastest_str:>25}  │")
print(f"│  {'Slowest Replier':<26}{slowest_str:>25}  │")

# Message Count
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│{'MESSAGE COUNT PER PERSON':^{WIDTH-2}}│")
print("├" + "─" * (WIDTH - 2) + "┤")
for name, c in sorted_users:
    item_str = "• " + name
    print(f"│  {item_str:<26}{c:>25}  │")

# Silent Streaks
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│{'LONGEST SILENT STREAKS':^{WIDTH-2}}│")
print("├" + "─" * (WIDTH - 2) + "┤")
for p, streak_str, _ in streak_results:
    item_str = "» " + p
    print(f"│  {item_str:<26}{streak_str:>25}  │")

# Top Words
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│{'TOP 5 FREQUENTLY USED WORDS':^{WIDTH-2}}│")
print("├" + "─" * (WIDTH - 2) + "┤")
for word, c in top10[:5]:
    item_str = "» " + word
    print(f"│  {item_str:<26}{c:>25}  │")

# Activity Heatmap
# Activity Heatmap - printing only (mat already built in Feature 4)
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│{'ACTIVITY HEATMAP (MESSAGES BY HOUR BLOCK)':^{WIDTH-2}}│")
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│  {'':6}{'00':3}{'03':3}{'06':3}{'09':3}{'12':3}{'15':3}{'18':3}{'21':3}{'':10}             │")
print("│" + " " * (WIDTH - 2) + "│")

for i in range(len(users)):
    block_totals = []
    for j in range(0, 24, 3):
        val = mat[i][j] + mat[i][j+1] + mat[i][j+2]
        block_totals.append(val)

    maxv = max(block_totals)
    row_chars = ""

    for val in block_totals:
        ratio = val / maxv if maxv > 0 else 0
        if ratio <= 0.25:
            ch = "."
        elif ratio <= 0.50:
            ch = "░"
        elif ratio <= 0.75:
            ch = "▒"
        else:
            ch = "█"
        row_chars = row_chars + ch + "  "

    name_part = f"{users[i][:6]:<6}"
    heat_part = f"{row_chars:<24}"
    line = "│  " + name_part + heat_part
    line = line + " " * (WIDTH - 1 - len(line)) + "│"
    print(line)

print("│" + " " * (WIDTH - 2) + "│")
# Personality Snapshot
print("├" + "─" * (WIDTH - 2) + "┤")
print(f"│{'PERSONALITY SNAPSHOT':^{WIDTH-2}}│")
print("├" + "─" * (WIDTH - 2) + "┤")
for p in users:
    arch = final_archetype[p][0]
    print(f"│  {p:<26}{arch:>25}  │")

print("└" + "─" * (WIDTH - 2) + "┘")
print(f"#{'THANKS FOR RUNNING GROUPDNA':^55}#")

#used chatpt and gemini for formatting and to solve allignment issues

┌───────────────────────────────────────────────────────┐
│                       GROUPDNA                        │
│          Your WhatsApp group chat, decoded.           │
│      Spotify Wrapped, but for your friend group!      │
│            WHATSAPP CHAT ANALYTICS RECEIPT            │
│          -----------------------------------          │
│     GROUP NAME : Hostel Bois                          │
├───────────────────────────────────────────────────────┤
│  METRIC / ITEM                         VALUE / STATS  │
├───────────────────────────────────────────────────────┤
│  Total Messages                                 3174  │
│  Total Participants                                6  │
│  Total Active Days                                60  │
│  Busiest Day                                04/05/24  │
│  Busiest Hour Block                    18:00 - 19:00  │
├───────────────────────────────────────────────────────┤
│                   RESPONSE PATTERNS                   │
├─────────────

**REFLECTION**

This project was a lot more challenging than I expected going in.The trickiest part for me was Feature 7.My first instinct was to use an if-elif chain for the archetypes which felt completely logical at the time — but it was wrong because it assigned whichever label matched first, not whichever fit best.Feature 6 also caught me off guard.I kept getting the current streak and max streak mixed up inside the while loop which gave wrong numbers with no error to help me figure out what was off.

Overall I learned that clean logic matters more than clever code, planning which variables need to survive across cells from the very beginning saves a lot of pain later.Also while doing each part it felt like revision of every concept and i used llms to debug and to understand logics and sometimes to just brainstorm logics too(chatgpt,claude,gemini)